In [ ]:
###################################
#
# AUTOENCODERS
#
###################################
#
# Manifold hypothesis  - real data lives on a low-dimensional manifold embedded in high-dimensional space
# PCA finds this manifold but only linear ones
# The architecture d -> k -> d with k <<< d creates an information bottleneck.
# GD minimising reconstruction error is forced to find an encoding tha preserves what matters most for reconstruction
# No explicit instruction about wht to preserver - emerges from loss landscape on optimisation
# Encoder and decoder co-evolve
#
# Linear AE = PCA - minimising residual after projection to a rank-k subspace which optimally is the top k eigenvectors of the data covariance matrix
# the weights obtained via GD wont equal PCA but the subspace spanned will be identical
#
# Used for:
# Anomaly detection
# Representation learning - encoder features
# Data compression
# Denoising - train on corrupted inputs, reconstruct clean forcing it to learn structure
#
# Fails when:
# k > d , it learns identity
# Decoder too expensive, stops being a translator nd becomes a lookup storage unit
# When layers before the bottleneck are too wide - unconstrained capacity, it can assign a unique, high magnitude signal to every training sample, it learns to multiplex the input - fix with l2\


In [ ]:
import numpy as np

class Autoencoder:
    def __init__(self,input_dim=784,lr=0.01,reg_str=0.0):
        self.lr = lr
        self.reg_str = reg_str
        self.W1 = np.random.randn(2,input_dim) * np.sqrt(2.0/input_dim)
        self.b1 = np.zeros(2)
        self.W2 = np.random.randn(2,input_dim) 
        self.b2 = np.zeros(input_dim)

    def relu(self,x):
        return np.maximum(0,x)
    def relu_derivative(self,x):
        return (x>0).astype(np.float32)
    def mse_loss(self,x,x_cap):
        return np.mean((x-x_cap)**2)

    def forward(self,x):
        
        self.x = x
        self.z = x @ self.W1.T + self.b1
        # self.a1 = self.relu(self.z)
        self.x_cap = self.z @ self.W2 + self.b2
        return self.x_cap
    
    def backward(self,x_cap,x):
        n = x.shape[0]
        dxcap = (2.0/n) * (x_cap-x)
        db2 = np.sum(dxcap,axis=0)
        dW2 = self.z.T @ dxcap
        dz = dxcap @ self.W2.T
        db1 = np.sum(dz,axis=0)
        # dz = dz * self.relu_derivative(self.z)
        dW1 = dz.T @ self.x
        for g in [dW1,dW2,db1,db2]:
            np.clip(g,-1.0,1.0,out=g)
        self.W2 -= self.lr * dW2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.b2 -= self.lr * db2
    def fit(self,x,epochs):
        n = x.shape[0]
        losses = []
        for epoch in range(epochs):
            x_cap = self.forward(x)
            loss = self.mse_loss(x,x_cap)
            losses.append(loss)
            self.backward(x_cap,x)
            if epoch % 200 == 0:
                print(f"[x] Epoch : {epoch} | Loss  : {loss}")
        return losses
    
# adding relu at the bottleneck level destroys representational capacity\
# x → linear → 2 → linear → x_cap (linear)
# Fix - add nonlinearity in the compression/decompression stages

In [ ]:
# inp flatten -> 784 -> 128 -> relu -> bottleneck -> 128 -> relu -> 784 
class Autoencoder_4:
    def __init__(self,input_dim=784,hidden_dim=128,bottleneck_dim=8,lr=0.001):
        self.lr = lr
        self.W1 = np.random.randn(hidden_dim,input_dim) * np.sqrt(2.0/input_dim)
        self.b1 = np.zeros(hidden_dim)

        self.W2 = np.random.randn(bottleneck_dim,hidden_dim) * np.sqrt(2.0/hidden_dim)
        self.b2 = np.zeros(bottleneck_dim)

        self.W3 = np.random.randn(hidden_dim,bottleneck_dim) * np.sqrt(2.0/bottleneck_dim)
        self.b3 = np.zeros(hidden_dim)

        self.W4 = np.random.randn(input_dim,hidden_dim) * np.sqrt(2.0/hidden_dim)
        self.b4 = np.zeros(input_dim)

    def relu(self,z):
        return np.maximum(0,z)
    def relu_derivative(self,z):
        return (z>0).astype(np.float32)
    def sigmoid(self,z):
        return 1 / (1 + np.exp(-z))
    
    def bce_loss(self,x,x_recon):
        eps = 1e-8
        return -np.mean(x * np.log(x_recon + eps) + (1-x) * np.log(1-x_recon + eps))
    def forward(self,x):
        self.x = x

        self.z1 = x @ self.W1.T + self.b1
        self.a1 = self.relu(self.z1)

        self.z2 = self.a1 @ self.W2.T + self.b2
        # no activation at bottleneck lvl

        self.z3 = self.z2 @ self.W3.T + self.b3
        self.a3 = self.relu(self.z3)

        self.z4 = self.a3 @ self.W4.T + self.b4
        self.x_recon = self.sigmoid(self.z4)
        return  self.x_recon
    
    def backward(self,x_recon,x):

        n = x.shape[0]

        dxrec = (x_recon-x)/n

        db4 = np.sum(dxrec,axis=0)
        dW4 = dxrec.T @ self.a3 
        da3 = dxrec @ self.W4

        dz3 = da3 * self.relu_derivative(self.z3)
        db3 = np.sum(dz3,axis=0)
        dW3 = dz3.T @ self.z2
        dz2 = dz3 @ self.W3

        dW2 = dz2.T @ self.a1
        da1 = dz2 @ self.W2 
        db2 = np.sum(dz2,axis=0)

        dz1 = da1 * self.relu_derivative(self.z1)
        dW1 = dz1.T @ self.x
        db1 = np.sum(dz1,axis=0)
        
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr *db1

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr *db2
        
        self.W3 -= self.lr * dW3
        self.b3 -= self.lr *db3
        
        self.W4 -= self.lr * dW4
        self.b4 -= self.lr *db4

    # def fit(self,x,epochs=1000,batch_size=64):
    #     losses = []
    #     n = x.shape[0]

    #     for epoch in range(epochs):
    #         perm = np.random.permutation(n)
    #         x_shuffled = x[perm]
    #         ep_loss = 0
    #         for i in range(0,n,batch_size):
    #             batch = x_shuffled[i:i+batch_size]
    #             x_recon = self.forward(batch)
    #             loss = self.bce_loss(batch,x_recon)
    #             ep_loss += loss
    #             self.backward(x_recon,batch)
    #         ep_loss /= (n//batch_size)
    #         losses.append(ep_loss)

    #         if epoch % 200 ==0:
    #             print(f"[x] Epoch : {epoch} | Loss  : {ep_loss}")
    #     return losses
    def fit(self,x,epochs=1000,batch_size=64,noise_factor=0.3):
        losses = []
        n = x.shape[0]

        for epoch in range(epochs):
            perm = np.random.permutation(n)
            x_shuffled = x[perm]
            ep_loss = 0

            for i in range(0,n,batch_size):
                clean = x_shuffled[i:i+batch_size]

                noisy = clean + noise_factor * np.random.randn(*clean.shape)
                noisy = np.clip(noisy, 0, 1)

                x_recon = self.forward(noisy)
                loss = self.bce_loss(clean,x_recon)
                ep_loss += loss

                self.backward(x_recon,clean)

            ep_loss /= (n//batch_size)
            losses.append(ep_loss)

            if epoch % 200 ==0:
                print(f"[x] Epoch : {epoch} | Loss  : {ep_loss}")

        return losses

In [ ]:
from torchvision import datasets
import matplotlib.pyplot as plt

train_data = datasets.MNIST(root='../experiments/data',train=True)
test_data = datasets.MNIST(root='../experiments/data',train=False)
model = Autoencoder_4(input_dim=784,lr=0.001)

X_train = train_data.data.numpy().reshape(-1,784) / 255.0
X_test = test_data.data.numpy().reshape(-1,784) / 255.0

losses = model.fit(X_train[:2000],epochs=1000)

In [ ]:
def add_noise(x, noise_factor=0.3):
    noisy = x + noise_factor * np.random.randn(*x.shape)
    return np.clip(noisy, 0, 1)

X_sample = X_test[:10]
noisy_sample = add_noise(X_sample)

recon = model.forward(noisy_sample)

fig, axes = plt.subplots(3, 10, figsize=(8,4))

for i in range(10):
    axes[0,i].imshow(X_sample[i].reshape(28,28))
    axes[0,i].axis('off')

    axes[1,i].imshow(noisy_sample[i].reshape(28,28))
    axes[1,i].axis('off')

    axes[2,i].imshow(recon[i].reshape(28,28))
    axes[2,i].axis('off')

axes[0,0].set_ylabel("Original")
axes[1,0].set_ylabel("Noisy")
axes[2,0].set_ylabel("Reconstructed")

plt.show()

# Normal AE has a trivial path to low loss - learning a near identity in hidden layers, noise corruption closes that path, as noise is random each foward pass so it has to learn the underlying 
# structure of clean data.
# Idenity with noise would be f(x+ err) = x + err != x 

In [ ]:
idx = np.random.randint(0, X_test.shape[0])
original = X_test[idx].reshape(1, -1)
reconstructed = model.forward(original)
original_img = original[0].reshape(28, 28)

reconstructed_img = reconstructed[0].reshape(28, 28)


plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(original_img, cmap='gray')
plt.subplot(1, 2, 2)
plt.title("Reconstruction")
plt.imshow(reconstructed_img, cmap='gray')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def encode(model, x):
    z1 = x @ model.W1.T + model.b1
    a1 = np.maximum(0, z1)
    z2 = a1 @ model.W2.T + model.b2
    return z2

def decode(model, z):
    z3 = z @ model.W3.T + model.b3
    a3 = np.maximum(0, z3)
    z4 = a3 @ model.W4.T + model.b4
    return 1 / (1 + np.exp(-z4))

X = X_test[:2000]
y = test_data.targets.numpy()[:2000]

Z = encode(model, X)

Z_plot = Z[:, :2]

plt.figure(figsize=(4,3))
scatter = plt.scatter(Z_plot[:,0], Z_plot[:,1], c=y, cmap='tab10', s=10)
plt.colorbar(scatter)
plt.title("Latent Space (first 2 dims)")
plt.xlabel("z1")
plt.ylabel("z2")
plt.show()

grid_size = 10
z1_vals = np.linspace(Z_plot[:,0].min(), Z_plot[:,0].max(), grid_size)
z2_vals = np.linspace(Z_plot[:,1].min(), Z_plot[:,1].max(), grid_size)

fig, axes = plt.subplots(grid_size, grid_size, figsize=(8,8))

latent_dim = model.W2.shape[0]

for i, xi in enumerate(z1_vals):
    for j, yi in enumerate(z2_vals):
        z = np.zeros((1, latent_dim))
        z[0, 0] = xi
        z[0, 1] = yi
        img = decode(model, z).reshape(28,28)
        axes[i,j].imshow(img)
        axes[i,j].axis('off')

plt.suptitle("Latent Space Traversal")
plt.show()

In [ ]:
# AE latent space is unstructured - interpolation is likely accidental, you cant sample new digits, learned codes for data are abitrary
# Variational AE force the latent space to follow a prior (standard Gaussian) -> draw z ~ N(0,I) -> decode get digit
# Instead of encoding x to z, x is encoded to mean(x),std(x)
# z = mean + std . epsilon , epsilon ~ N(0,I)
# Loss changes to reconstruction_loss + KL ( N(mean,var) || N(0,I) )
